# Feature engineering: credit card fraud detection

This notebook covers SMOTE oversampling, feature scaling, interaction features, and time-based feature transformations to prepare data for modeling.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("../data/fraud_transactions.csv")
print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df['is_fraud'].mean():.4f}")

## Train/test split (stratified)

In [ ]:
feature_cols = [c for c in df.columns if c != "is_fraud"]
X = df[feature_cols].values.astype(float)
y = df["is_fraud"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples, fraud rate: {y_train.mean():.4f}")
print(f"Test set:     {X_test.shape[0]} samples, fraud rate: {y_test.mean():.4f}")
print(f"Train fraud count: {y_train.sum()}")
print(f"Test fraud count:  {y_test.sum()}")

## SMOTE oversampling

SMOTE (Synthetic Minority Over-sampling Technique) creates synthetic fraud examples by interpolating between existing fraud transactions in feature space. This balances the training set without simply duplicating records.

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {X_train.shape[0]} samples")
print(f"  Legitimate: {(y_train == 0).sum()}")
print(f"  Fraud:      {(y_train == 1).sum()}")
print(f"\nAfter SMOTE: {X_train_smote.shape[0]} samples")
print(f"  Legitimate: {(y_train_smote == 0).sum()}")
print(f"  Fraud:      {(y_train_smote == 1).sum()}")
print(f"  Ratio:      {(y_train_smote == 0).sum() / (y_train_smote == 1).sum():.1f}:1")

In [ ]:
# Visualize class balance before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#3B6FD4", "#E8C230"]

before = pd.Series(y_train).value_counts().sort_index()
axes[0].bar(["Legitimate", "Fraud"], before.values, color=colors, edgecolor="black")
axes[0].set_title("Before SMOTE")
axes[0].set_ylabel("Count")
for i, v in enumerate(before.values):
    axes[0].text(i, v + 50, f"{v:,}", ha="center", fontweight="bold")

after = pd.Series(y_train_smote).value_counts().sort_index()
axes[1].bar(["Legitimate", "Fraud"], after.values, color=colors, edgecolor="black")
axes[1].set_title("After SMOTE")
axes[1].set_ylabel("Count")
for i, v in enumerate(after.values):
    axes[1].text(i, v + 50, f"{v:,}", ha="center", fontweight="bold")

plt.suptitle("Class distribution before and after SMOTE", fontsize=13)
plt.tight_layout()
plt.show()

## Feature scaling

StandardScaler centers features to mean=0 and std=1. We also compare with RobustScaler which is less sensitive to outliers (useful for amount and distance features with heavy tails).

In [ ]:
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

X_train_standard = standard_scaler.fit_transform(X_train_smote)
X_train_robust = robust_scaler.fit_transform(X_train_smote)

# Compare distributions before and after scaling
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
feat_idx = 0  # amount

axes[0].hist(X_train_smote[:, feat_idx], bins=50, color="#3B6FD4", edgecolor="black", alpha=0.7)
axes[0].set_title(f"Original: {feature_cols[feat_idx]}")

axes[1].hist(X_train_standard[:, feat_idx], bins=50, color="#E8C230", edgecolor="black", alpha=0.7)
axes[1].set_title(f"StandardScaler: {feature_cols[feat_idx]}")

axes[2].hist(X_train_robust[:, feat_idx], bins=50, color="#22c55e", edgecolor="black", alpha=0.7)
axes[2].set_title(f"RobustScaler: {feature_cols[feat_idx]}")

plt.suptitle("Effect of scaling on amount feature", fontsize=13)
plt.tight_layout()
plt.show()

## Interaction features

Creating interaction terms to capture non-linear relationships between features that may signal fraud.

In [ ]:
df_eng = df.copy()

# Interaction: amount * ratio_to_median (high amount AND unusual ratio)
df_eng["amount_x_ratio"] = df_eng["amount"] * df_eng["ratio_to_median_purchase"]

# Interaction: distance_from_home * distance_from_last (both large = suspicious)
df_eng["dist_home_x_dist_last"] = df_eng["distance_from_home"] * df_eng["distance_from_last_transaction"]

# Interaction: night * amount (large night transactions)
df_eng["night_x_amount"] = df_eng["is_night"] * df_eng["amount"]

# Velocity ratio: hourly velocity relative to daily
df_eng["velocity_ratio"] = np.where(
    df_eng["num_transactions_last_day"] > 0,
    df_eng["num_transactions_last_hour"] / df_eng["num_transactions_last_day"],
    df_eng["num_transactions_last_hour"]
)

# Log amount (reduces skewness)
df_eng["log_amount"] = np.log1p(df_eng["amount"])

new_features = ["amount_x_ratio", "dist_home_x_dist_last", "night_x_amount", "velocity_ratio", "log_amount"]
print("New interaction features:")
for feat in new_features:
    corr_fraud = df_eng[feat].corr(df_eng["is_fraud"])
    print(f"  {feat}: corr with fraud = {corr_fraud:.4f}")

In [ ]:
# Visualize interaction features by class
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flat

for i, feat in enumerate(new_features):
    for label, color, name in [(0, "#3B6FD4", "Legitimate"), (1, "#E8C230", "Fraud")]:
        subset = df_eng[df_eng["is_fraud"] == label]
        axes[i].hist(subset[feat], bins=40, alpha=0.6, label=name, color=color, edgecolor="black")
    axes[i].set_title(feat)
    axes[i].legend()

axes[-1].set_visible(False)
plt.suptitle("Interaction feature distributions by class", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Time-based features

Cyclical encoding for hour-of-day using sine and cosine transforms, which preserves the circular nature of time (hour 23 is close to hour 0).

In [ ]:
# Cyclical encoding for time_hour
df_eng["hour_sin"] = np.sin(2 * np.pi * df_eng["time_hour"] / 24)
df_eng["hour_cos"] = np.cos(2 * np.pi * df_eng["time_hour"] / 24)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: sin vs cos by class
for label, color, name in [(0, "#3B6FD4", "Legit"), (1, "#E8C230", "Fraud")]:
    subset = df_eng[df_eng["is_fraud"] == label].sample(min(500, len(df_eng[df_eng["is_fraud"] == label])), random_state=42)
    axes[0].scatter(subset["hour_sin"], subset["hour_cos"], alpha=0.4, s=15, c=color, label=name)
axes[0].set_xlabel("hour_sin")
axes[0].set_ylabel("hour_cos")
axes[0].set_title("Cyclical time encoding")
axes[0].legend()

# Correlation comparison: raw vs cyclical
time_features = ["time_hour", "hour_sin", "hour_cos"]
corrs = [df_eng[f].corr(df_eng["is_fraud"]) for f in time_features]
axes[1].barh(time_features, [abs(c) for c in corrs], color=["#3B6FD4", "#E8C230", "#22c55e"], edgecolor="black")
axes[1].set_xlabel("|Correlation with is_fraud|")
axes[1].set_title("Time feature correlation with fraud")
for i, (feat, c) in enumerate(zip(time_features, corrs)):
    axes[1].text(abs(c) + 0.002, i, f"{c:.4f}", va="center")

plt.tight_layout()
plt.show()

## Updated correlation heatmap with engineered features

In [ ]:
corr_eng = df_eng.corr()

# Show only correlations with is_fraud, sorted
fraud_corr = corr_eng["is_fraud"].drop("is_fraud").sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#E8C230" if v > 0 else "#3B6FD4" for v in fraud_corr.values]
ax.barh(fraud_corr.index, fraud_corr.values, color=colors, edgecolor="black")
ax.set_xlabel("Correlation with is_fraud")
ax.set_title("Feature correlation with fraud target (including engineered features)")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("Top 5 positive correlations with fraud:")
print(fraud_corr.head().round(4))
print("\nTop 5 negative correlations with fraud:")
print(fraud_corr.tail().round(4))

## Feature distribution after SMOTE

Verifying that SMOTE-generated samples maintain realistic feature distributions.

In [ ]:
# Compare original fraud vs SMOTE-generated fraud
df_smote = pd.DataFrame(X_train_smote, columns=feature_cols)
df_smote["is_fraud"] = y_train_smote

original_fraud = df[df["is_fraud"] == 1]
smote_fraud = df_smote[df_smote["is_fraud"] == 1]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flat
check_features = ["amount", "distance_from_home", "ratio_to_median_purchase",
                   "num_transactions_last_hour", "distance_from_last_transaction", "time_hour"]

for i, feat in enumerate(check_features):
    axes[i].hist(original_fraud[feat], bins=30, alpha=0.6, label="Original fraud", color="#E8C230", edgecolor="black", density=True)
    axes[i].hist(smote_fraud[feat], bins=30, alpha=0.4, label="SMOTE fraud", color="#3B6FD4", edgecolor="black", density=True)
    axes[i].set_title(feat)
    axes[i].legend()

plt.suptitle("Original fraud vs SMOTE-generated fraud distributions", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Summary

Feature engineering steps applied:

1. **SMOTE oversampling** -- balanced the training set from 49:1 to 1:1 ratio, generating synthetic fraud samples
2. **StandardScaler** -- normalized features for logistic regression; tree-based models do not require scaling
3. **Interaction features** -- amount x ratio, distance products, and night x amount capture non-linear fraud signals
4. **Cyclical time encoding** -- sine/cosine transforms preserve the circular nature of hour-of-day
5. **Log transform** -- log(amount) reduces right skewness in the transaction amount distribution
6. **Velocity ratio** -- hourly-to-daily transaction ratio captures burst activity patterns

Note: For the modeling notebooks, we use the base 10 features without interaction terms, as the tree-based models (XGBoost, LightGBM) can learn these interactions internally. The interaction features are demonstrated here to show the feature engineering thought process.